---
## Setup: Start vLLM + Onboard OpenClaw (one cell does everything)

Run the cell below **once**. It will:
1. Start vLLM serving Qwen3-4B on your AMD GPU (background)
2. Tail the startup logs live until the model is loaded
3. Onboard OpenClaw against the local vLLM endpoint
4. Activate the credential and set lean mode

When it finishes (green), proceed below to chat with your agent directly in this notebook.

In [ ]:
import subprocess, time, os, re, sys
try:
    import httpx
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "httpx"])
    import httpx

# ── 1. Start vLLM in background ─────────────────────────────────────
print("▶ Starting vLLM server...")
env = {**os.environ, "VLLM_USE_TRITON_FLASH_ATTN": "0"}
vllm_proc = subprocess.Popen(
    ["vllm", "serve", "Qwen/Qwen3-4B",
     "--served-model-name", "Qwen3-4B",
     "--api-key", "abc-123",
     "--port", "8000",
     "--enable-auto-tool-choice",
     "--tool-call-parser", "hermes",
     "--trust-remote-code",
     "--max_model_len", "24272",
     "--gpu-memory-utilization", "0.85"],
    stdout=open("/tmp/vllm.log", "w"),
    stderr=subprocess.STDOUT,
    env=env,
)
print(f"  PID {vllm_proc.pid} launched, waiting for model load...\n")

# ── 2. Tail vLLM log + poll /health until ready ─────────────────────
seen = 0
ready = False
for tick in range(300):  # up to ~5 min
    time.sleep(1)
    try:
        with open("/tmp/vllm.log") as f:
            lines = f.readlines()
        new_lines = lines[seen:]
        if new_lines:
            for ln in new_lines[-5:]:  # show last 5 new lines
                print(ln.rstrip())
            seen = len(lines)
    except FileNotFoundError:
        pass
    # Check health every 5s
    if tick % 5 == 4:
        try:
            r = httpx.get("http://localhost:8000/health", timeout=2)
            if r.status_code == 200:
                ready = True
                break
        except Exception:
            pass

if not ready:
    print("\n✗ vLLM did not become healthy in 5 minutes. Check /tmp/vllm.log")
    raise SystemExit(1)

print("\n✓ vLLM is healthy!\n")

# ── 3. Onboard OpenClaw ──────────────────────────────────────────────
print("▶ Onboarding OpenClaw...")
r = subprocess.run([
    "openclaw", "onboard", "--non-interactive", "--mode", "local",
    "--auth-choice", "custom-api-key",
    "--custom-base-url", "http://localhost:8000/v1",
    "--custom-model-id", "Qwen3-4B", "--custom-provider-id", "vllm",
    "--custom-compatibility", "openai", "--custom-api-key", "abc-123",
    "--secret-input-mode", "plaintext", "--custom-image-input",
    "--skip-channels", "--skip-search", "--skip-hooks", "--skip-ui",
    "--skip-health", "--accept-risk",
], capture_output=True, text=True)
print(r.stdout.strip())
if r.stderr.strip():
    print(r.stderr.strip())

# Activate the latest credential
match = re.search(r"(vllm:setup-[\w-]+)", r.stdout + r.stderr)
if match:
    cred_id = match.group(1)
    print(f"\n▶ Activating credential {cred_id}...")
    a = subprocess.run(
        ["openclaw", "models", "auth", "activate", cred_id, "--agent", "main"],
        capture_output=True, text=True,
    )
    print(a.stdout.strip())

# ── 4. Lean mode config ─────────────────────────────────────────────
print("\n▶ Setting lean mode...")
subprocess.run(
    ["openclaw", "config", "set",
     "agents.defaults.experimental.localModelLean", "true"],
    capture_output=True, text=True,
)
print("  Config applied.")

print("\n" + "=" * 60)
print("✓ All done! Run the cells below to chat with your agent.")
print("=" * 60)

---
## Chat with OpenClaw

Run the cell below to open an embedded chat interface. Type your message in the input box and press **Enter** or click **Send**. The response from Qwen3-4B (running on your MI300X GPU) will appear in the chat window.

Try saying **Hi** first, then ask it anything you like!

In [ ]:
import re, threading, time
try:
    import pexpect
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pexpect"])
    import pexpect
try:
    import ipywidgets as widgets
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ipywidgets"])
    import ipywidgets as widgets
from IPython.display import display

# ── Spawn openclaw tui in a pseudo-terminal ──────────────────────────
_tui = pexpect.spawn("openclaw tui", encoding="utf-8", timeout=300)
_tui.setwinsize(50, 120)  # rows, cols

# Strip ANSI escape sequences for clean display
_ANSI_RE = re.compile(r"\x1b\[[0-9;]*[a-zA-Z]|\x1b\].*?\x07|\x1b\[.*?[@-~]")

def _clean(text):
    return _ANSI_RE.sub("", text).strip()

# Wait for the TUI to be ready (look for the input prompt)
print("Starting openclaw tui... (waiting for prompt)")
try:
    _tui.expect([r"[>›❯\$]", pexpect.TIMEOUT], timeout=30)
    print("✓ openclaw tui is ready!")
except Exception:
    print("⚠ TUI may still be loading — first message might take a moment.")

# ── Widgets ──────────────────────────────────────────────────────────
chat_log = widgets.Output(layout=widgets.Layout(
    width="100%", min_height="350px", max_height="600px",
    overflow_y="auto", border="1px solid #444", padding="8px",
))
user_input = widgets.Text(
    placeholder="Type your message and press Enter or click Send...",
    layout=widgets.Layout(width="85%"),
)
send_btn = widgets.Button(
    description="Send", button_style="primary",
    layout=widgets.Layout(width="14%"),
)

_lock = threading.Lock()

def send(msg=None):
    """Send a message to openclaw tui and capture the response."""
    if msg is None:
        msg = user_input.value.strip()
    if not msg:
        return
    user_input.value = ""

    with chat_log:
        print(f"\n{'─'*60}")
        print(f"YOU:  {msg}")
        print(f"{'─'*60}")

    with _lock:
        try:
            _tui.sendline(msg)
            # Read output until we see the next prompt or a long pause
            chunks = []
            while True:
                try:
                    idx = _tui.expect([r"[>›❯\$]", pexpect.TIMEOUT], timeout=60)
                    chunks.append(_tui.before)
                    if idx == 0:  # got the prompt back — response complete
                        break
                    # timeout — grab what we have and break
                    break
                except pexpect.EOF:
                    chunks.append(_tui.before)
                    break

            raw = "".join(chunks)
            reply = _clean(raw)
            # Remove the echo of our own message from the start
            if reply.startswith(msg):
                reply = reply[len(msg):].strip()

            with chat_log:
                print(f"\nOPENCLAW:")
                for line in reply.splitlines():
                    cleaned = line.strip()
                    if cleaned:
                        print(f"  {cleaned}")
                print()
        except Exception as e:
            with chat_log:
                print(f"\nERROR: {e}\n")

def _on_click(btn):
    send()

def _on_submit(change):
    send()

send_btn.on_click(_on_click)
user_input.on_submit(_on_submit)

display(widgets.VBox([
    chat_log,
    widgets.HBox([user_input, send_btn], layout=widgets.Layout(width="100%", gap="1%")),
]))

---
## Try it: Clone a Repo and Ask for Analysis

Run the cell below — it clones a public repo and sends an analysis request through the chat above. Watch the response appear in the chat window.

In [ ]:
import subprocess, os

REPO_URL  = "https://github.com/matrived/blr_workshop.git"
CLONE_DIR = "/workspace/blr_workshop"

# Clone the repo (skip if already cloned)
if not os.path.isdir(CLONE_DIR):
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, CLONE_DIR], check=True)
    print("Done.\n")
else:
    print(f"{CLONE_DIR} already exists, skipping clone.\n")

# Send the analysis prompt through the chat widget
send(
    "Clone https://github.com/matrived/blr_workshop.git in /workspace directory, "
    "show me the repository structure line-by-line one-liner description of them "
    "and also describe cell-by-cell what "
    "unsloth-finetuning/grpo_game2048-finetune-unsloth/"
    "gpt_oss_20B_Reinforcement_Learning_2048_Game_BF16.ipynb "
    "file is doing in each cell in 1 line and lastly what could be improved"
)